### Import thư viện

In [14]:
from collections import deque
import random

### Trạng thái mục tiêu

In [15]:
goal_state = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 0]
]

### Random trạng thái ban đầu

In [33]:
def input_state():
    numbers = list(range(9))  
    random.shuffle(numbers)

    state = [
        numbers[0:3],
        numbers[3:6],
        numbers[6:9]
    ]

    return state

### Các hàm xử lý trạng thái

In [17]:
def copy_state(state):
    return [row[:] for row in state]

In [18]:
def print_state(state):
    for row in state:
        print(row)

In [19]:
def state_key(state):
    return tuple(tuple(row) for row in state)

In [20]:
def find_zero(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j
    return None, None

In [21]:
def is_goal_state(state):
    return state_key(state) == state_key(goal_state)

### Các hành động có thể đi

In [22]:
def get_possible_moves(state):
    zero_x, zero_y = find_zero(state)

    possible_moves = []

    if zero_x > 0:
        possible_moves.append("UP")

    if zero_x < 2:
        possible_moves.append("DOWN")

    if zero_y > 0:
        possible_moves.append("LEFT")

    if zero_y < 2:
        possible_moves.append("RIGHT")

    return possible_moves

### Model: tạo trạng thái mới sau khi đi một hành động

In [23]:
def model(state, action):
    if action is None:
        return state

    new_state = copy_state(state)
    zero_x, zero_y = find_zero(new_state)

    new_x, new_y = zero_x, zero_y

    if action == "UP":
        new_x = zero_x - 1
    elif action == "DOWN":
        new_x = zero_x + 1
    elif action == "LEFT":
        new_y = zero_y - 1
    elif action == "RIGHT":
        new_y = zero_y + 1

    new_state[zero_x][zero_y], new_state[new_x][new_y] = (
        new_state[new_x][new_y],
        new_state[zero_x][zero_y]
    )

    return new_state

## Breadth-First-Search

### Node

In [24]:
def create_node(state, parent=None, action=None, depth=0):
    node = {
        "state": state,
        "parent": parent,
        "action": action,
        "depth": depth
    }

    return node

### Expand

In [25]:
def expand(node):
    children = []

    state = node["state"]
    possible_moves = get_possible_moves(state)

    for action in possible_moves:
        new_state = model(state, action)

        child = create_node(
            state=new_state,
            parent=node,
            action=action,
            depth=node["depth"] + 1
        )

        children.append(child)

    return children

### Breadth-First-Search

In [37]:
def breadth_first_search(initial_state):
    node = create_node(initial_state)

    if is_goal_state(node["state"]):
        return node

    frontier = deque([node])
    reached = set()
    reached.add(state_key(initial_state))

    while len(frontier) > 0:
        node = frontier.popleft()

        children = expand(node)

        # Bước 1: kiểm tra trước xem trong các trạng thái con có goal không
        for child in children:
            s = child["state"]

            if is_goal_state(s):
                return child

        # Bước 2: nếu không có goal thì mới thêm các trạng thái chưa đi vào frontier
        for child in children:
            s = child["state"]
            key = state_key(s)

            if key not in reached:
                reached.add(key)
                frontier.append(child)

    return None

### In đường đi lời giải

In [27]:
def get_solution_path(solution_node):
    path = []

    node = solution_node

    while node is not None:
        path.append(node)
        node = node["parent"]

    path.reverse()

    return path

In [28]:
def print_solution(solution_node):
    path = get_solution_path(solution_node)

    for i, node in enumerate(path):
        print(f"\n========== BƯỚC {i} ==========")

        if node["action"] is not None:
            print("Hành động:", node["action"])

        print_state(node["state"])

### Main program

In [40]:
initial_state = input_state()

print("\nTrạng thái ban đầu:")
print_state(initial_state)

print("\nTrạng thái đích cần đạt:")
print_state(goal_state)

solution_node = breadth_first_search(initial_state)

if solution_node is None:
    print("\nDỪNG LẠI!")
    print("Lý do: Không tìm thấy lời giải.")
else:
    print("\nTÌM THẤY LỜI GIẢI!")
    print("Số bước đi ít nhất:", solution_node["depth"])

    print_solution(solution_node)


Trạng thái ban đầu:
[0, 1, 6]
[4, 8, 2]
[7, 3, 5]

Trạng thái đích cần đạt:
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]

TÌM THẤY LỜI GIẢI!
Số bước đi ít nhất: 14

========== BƯỚC 0 ==========
[0, 1, 6]
[4, 8, 2]
[7, 3, 5]

========== BƯỚC 1 ==========
Hành động: RIGHT
[1, 0, 6]
[4, 8, 2]
[7, 3, 5]

========== BƯỚC 2 ==========
Hành động: RIGHT
[1, 6, 0]
[4, 8, 2]
[7, 3, 5]

========== BƯỚC 3 ==========
Hành động: DOWN
[1, 6, 2]
[4, 8, 0]
[7, 3, 5]

========== BƯỚC 4 ==========
Hành động: LEFT
[1, 6, 2]
[4, 0, 8]
[7, 3, 5]

========== BƯỚC 5 ==========
Hành động: DOWN
[1, 6, 2]
[4, 3, 8]
[7, 0, 5]

========== BƯỚC 6 ==========
Hành động: RIGHT
[1, 6, 2]
[4, 3, 8]
[7, 5, 0]

========== BƯỚC 7 ==========
Hành động: UP
[1, 6, 2]
[4, 3, 0]
[7, 5, 8]

========== BƯỚC 8 ==========
Hành động: LEFT
[1, 6, 2]
[4, 0, 3]
[7, 5, 8]

========== BƯỚC 9 ==========
Hành động: UP
[1, 0, 2]
[4, 6, 3]
[7, 5, 8]

========== BƯỚC 10 ==========
Hành động: RIGHT
[1, 2, 0]
[4, 6, 3]
[7, 5, 8]

========== BƯỚC 11 =========